<a href="https://colab.research.google.com/github/mohamadfaisalbashir/Practical-Linear-Algebra/blob/main/10_Row_Reduction_and_LU_Decomposition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Row Reduction and LU Decomposition**

This notebook covers Row Reduction and LU Decomposition:
1. Systems of Equations
2. Row Reduction and Echelon Form
3. LU Decomposition
4. Summary

## **1. Systems of Equations**

Before we can understand LU decomposition, we need to develop a solid understanding of systems of equations and how they relate to matrices. A system of equations is a collection of multiple equations with multiple unknowns that must all be satisfied simultaneously. For example, consider a system with two equations and two unknowns: x + y = 4 and -x/2 + y = 2. The challenge is to find the values of x and y that satisfy both equations at the same time. In higher dimensions, systems can involve many equations and many unknowns, making them impossible to solve by hand using traditional algebraic substitution. The matrix formulation of a system of equations provides a powerful framework for solving such systems efficiently. A system of equations can be translated into the matrix equation Ax = b, where A is a matrix containing the coefficients of the variables, x is a column vector containing the unknown variables we want to solve for, and b is a column vector containing the constants on the right-hand side of the equations. This matrix formulation not only provides a compact representation but also unlocks powerful linear algebra techniques for solving the system. The process of converting a system of equations into matrix form involves organizing the equations so constants are on the right side and coefficients are organized systematically into a matrix, with each row corresponding to one equation and each column corresponding to one variable. This systematic approach makes the system amenable to computational methods like row reduction and LU decomposition.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
import sympy as sym
import warnings
warnings.filterwarnings('ignore')

### **1.1 Converting Equations into Matrices**

The process of converting a system of equations into matrix form is systematic and straightforward. The key steps are: (1) organize all equations so that variables with their coefficients are on the left side and constants are on the right side, (2) arrange variables in the same order across all equations (all equations should have x first, then y, then z, etc.), and (3) extract the coefficients into a matrix A, the variables into a column vector x, and the constants into a column vector b. Once the system is in the form Ax = b, you have unlocked the power of linear algebra for solving the system. Let's illustrate with the example system: x + y = 4 and -x/2 + y = 2. The coefficient matrix A would be [1, 1; -1/2, 1], the variable vector x would be [x; y], and the constant vector b would be [4; 2]. The matrix equation Ax = b is equivalent to the original system of equations, and solving this matrix equation is equivalent to solving the original system. This conversion is not just a notational convenience; it enables the use of sophisticated matrix algorithms to solve the system efficiently and numerically stably.

In [2]:
A = np.array([[1, 1],
              [-1/2, 1]])

# Constants vector b
b = np.array([4, 2])

print("System of equations:")
print("  x + y = 4")
print(" -x/2 + y = 2")

print("\nMatrix form Ax = b:")
print("\nA =")
print(A)
print("\nb =")
print(b)

# Solve using numpy
x = np.linalg.solve(A, b)
print(f"\nSolution: x = {x[0]:.4f}, y = {x[1]:.4f}")

# Verify the solution
verification = A @ x
print(f"\nVerification (Ax should equal b):")
print(f"Ax = {verification}")
print(f"b = {b}")
print(f"Error: {np.linalg.norm(verification - b):.2e}")

System of equations:
  x + y = 4
 -x/2 + y = 2

Matrix form Ax = b:

A =
[[ 1.   1. ]
 [-0.5  1. ]]

b =
[4 2]

Solution: x = 1.3333, y = 2.6667

Verification (Ax should equal b):
Ax = [4. 2.]
b = [4 2]
Error: 2.22e-16


## **2. Row Reduction and Echelon Form**

Row reduction is a systematic procedure for transforming a matrix into a special form called echelon form through a series of elementary row operations. The goal of row reduction is to transform a dense matrix into an upper-triangular matrix. The elementary row operations allowed are: (1) multiply a row by a nonzero scalar, (2) add a multiple of one row to another row, and (3) swap two rows. These operations are allowed because they do not change the solution to the system of equations (in the case of augmented matrices). A matrix is said to be in echelon form (also called row echelon form) if: (1) the leftmost nonzero element in each row (called the pivot) is to the right of the pivot in the row above, and (2) any rows consisting entirely of zeros are located below all nonzero rows. A matrix in echelon form is upper-triangular, meaning all elements below the diagonal are zero. The process of row reduction can be viewed as a sequence of linear transformations that can themselves be represented by matrices. Specifically, if we denote by L^(-1) the matrix representing all the row operations, then the transformation can be written as L^(-1)A = U, where U is the upper-triangular echelon form. This relationship is crucial because it leads directly to the concept of LU decomposition, where L^(-1) is inverted to give L, resulting in the factorization A = LU.

In [3]:
# Example of row reduction
# Original system (augmented matrix)
A_augmented = np.array([[1, 1, 4],
                        [-1/2, 1, 2]], dtype=float)

print("Original augmented matrix [A|b]:")
print(A_augmented)

# Row operation: R2 = R2 + (1/2)R1
factor = A_augmented[1, 0] / A_augmented[0, 0]
A_augmented[1, :] = A_augmented[1, :] - factor * A_augmented[0, :]

print("\nAfter row operation R2 = R2 - (-0.5)*R1:")
print(A_augmented)
print("Matrix is now in echelon form (upper-triangular)")

Original augmented matrix [A|b]:
[[ 1.   1.   4. ]
 [-0.5  1.   2. ]]

After row operation R2 = R2 - (-0.5)*R1:
[[1.  1.  4. ]
 [0.  1.5 4. ]]
Matrix is now in echelon form (upper-triangular)


### **2.1 Gaussian Elimination**

Gaussian elimination is a classical algorithm for solving systems of linear equations that combines three key steps: (1) form an augmented matrix by attaching the constants vector b to the coefficient matrix A, (2) use row reduction to transform the augmented matrix into echelon form (upper-triangular), and (3) use back substitution to solve for the variables starting from the last equation and working backwards. The beauty of Gaussian elimination is that it transforms a system of equations with interwoven variables into a sequence of equations where each equation depends on fewer variables, making them progressively easier to solve. For example, if the original system involved x and y in both equations, after row reduction the second equation might only involve y, allowing you to solve for y directly and then substitute back to find x. Back substitution gets its name from the process of working backwards through the equations: first solve the last equation (which typically has only one unknown), then use that solution to solve the second-to-last equation, and so on. This process is computationally efficient and forms the basis for solving systems in both hand calculations and computer algorithms. While modern numerical software usually implements more sophisticated variants to ensure numerical stability, Gaussian elimination remains the conceptual foundation for understanding how systems of linear equations are solved.

In [4]:
# Complete Gaussian elimination example
A = np.array([[1, 1],
              [-1/2, 1]], dtype=float)
b = np.array([4, 2], dtype=float)

# Create augmented matrix
augmented = np.column_stack([A, b])

print("Original augmented matrix [A|b]:")
print(augmented)

# Row reduction to echelon form
m, n = augmented.shape
for i in range(m-1):
    # Find pivot
    pivot = augmented[i, i]
    if abs(pivot) < 1e-10:
        continue

    # Eliminate below
    for j in range(i+1, m):
        factor = augmented[j, i] / pivot
        augmented[j, :] = augmented[j, :] - factor * augmented[i, :]

print("\nAfter forward elimination (echelon form):")
print(augmented)

# Back substitution
x = np.zeros(m)
for i in range(m-1, -1, -1):
    x[i] = augmented[i, n-1]
    for j in range(i+1, m):
        x[i] -= augmented[i, j] * x[j]
    x[i] /= augmented[i, i]

print(f"\nSolution from back substitution: x = {x}")

Original augmented matrix [A|b]:
[[ 1.   1.   4. ]
 [-0.5  1.   2. ]]

After forward elimination (echelon form):
[[1.  1.  4. ]
 [0.  1.5 4. ]]

Solution from back substitution: x = [1.33333333 2.66666667]


### **2.2 Gauss-Jordan Elimination and RREF**

Gauss-Jordan elimination is an extension of Gaussian elimination that goes beyond just creating an upper-triangular echelon form. Instead, it continues the row reduction process to create the reduced row echelon form (RREF), in which every pivot element is 1 and every pivot is the only nonzero element in its column. This means the RREF contains an identity matrix as a submatrix (typically in the upper left portion for augmented matrices). The process involves two stages: (1) first use standard row reduction to create upper-triangular echelon form (as in Gaussian elimination), then (2) work upward from the bottom row, eliminating all elements above each pivot to create the fully reduced form. The major advantage of RREF over the standard echelon form is that back substitution is not necessary; the solution to the system of equations can be read directly from the RREF. If the augmented matrix has been transformed to [I|x], then x is directly the solution vector. The RREF of a matrix is unique, meaning that every matrix has exactly one associated RREF (unlike echelon form, which is not unique). This uniqueness makes RREF particularly useful in theoretical analysis. While RREF is less commonly used in modern computational practice due to numerical stability concerns, understanding RREF provides valuable insight into how systems of equations are solved and how matrix properties relate to the solutions.

In [5]:
# Compute RREF using sympy
A = np.array([[1, 1, 4],
              [-1/2, 1, 2]], dtype=float)

# Convert to sympy matrix
sympy_mat = sym.Matrix(A)

# Compute RREF
rref_mat, pivot_cols = sympy_mat.rref()

print("Original augmented matrix [A|b]:")
print(A)

print("\nReduced Row Echelon Form (RREF):")
print(np.array(rref_mat).astype(float))

print("\nPivot columns:", pivot_cols)

print("\nFrom RREF, the solution can be read directly:")
print(f"x = {float(rref_mat[0, 2])}")
print(f"y = {float(rref_mat[1, 2])}")

Original augmented matrix [A|b]:
[[ 1.   1.   4. ]
 [-0.5  1.   2. ]]

Reduced Row Echelon Form (RREF):
[[1.         0.         1.33333333]
 [0.         1.         2.66666667]]

Pivot columns: (0, 1)

From RREF, the solution can be read directly:
x = 1.3333333333333333
y = 2.6666666666666665


## **3. LU Decomposition**

LU decomposition is one of the most important matrix factorizations in applied linear algebra. The "LU" stands for "lower upper," referring to the two triangular matrices that make up the decomposition. The goal of LU decomposition is to factor a matrix A into the product of two matrices: A = LU, where L is a lower-triangular matrix and U is an upper-triangular matrix. The key insight is that the row reduction process that transforms A into echelon form U can itself be represented as matrix multiplication by a series of transformation matrices. If we collect all these transformation matrices into L^(-1), then row reduction gives us L^(-1)A = U, or equivalently, A = L^(-1)^(-1) U = LU, where L is the inverse of L^(-1). The lower-triangular matrix L captures information about the row operations performed during the reduction, while the upper-triangular matrix U is the echelon form itself. One important constraint on L is that its diagonal elements are all 1; this ensures that LU decomposition is unique for a full-rank square matrix. The LU decomposition is tremendously useful because it allows us to solve systems of equations efficiently: given Ax = b, if we know A = LU, then LUx = b, which can be solved in two stages by first solving Ly = b (forward substitution, very fast for triangular L) and then solving Ux = y (back substitution, very fast for triangular U). This two-stage process is much more efficient than repeatedly solving systems for different right-hand sides because you only compute the LU decomposition once and then reuse it for multiple solves.

In [6]:
# Basic LU decomposition
A = np.array([[2, 2, 4],
              [1, 0, 3],
              [2, 1, 2]], dtype=float)

P, L, U = linalg.lu(A)

print("Original matrix A:")
print(A)

print("\nLower-triangular matrix L (with 1s on diagonal):")
print(L)

print("\nUpper-triangular matrix U:")
print(U)

print("\nPermutation matrix P:")
print(P)

# Verify PA = LU
reconstructed = P @ A
product = L @ U

print(f"\nVerification: PA - LU (should be near zero)")
print(f"Error: {np.linalg.norm(reconstructed - product):.2e}")

# Can also verify as A = P^T @ L @ U (since P^(-1) = P^T for permutation matrices)
A_reconstructed = P.T @ L @ U
print(f"\nVerification: A - P^T @ L @ U (should be near zero)")
print(f"Error: {np.linalg.norm(A - A_reconstructed):.2e}")

Original matrix A:
[[2. 2. 4.]
 [1. 0. 3.]
 [2. 1. 2.]]

Lower-triangular matrix L (with 1s on diagonal):
[[1.  0.  0. ]
 [0.5 1.  0. ]
 [1.  1.  1. ]]

Upper-triangular matrix U:
[[ 2.  2.  4.]
 [ 0. -1.  1.]
 [ 0.  0. -3.]]

Permutation matrix P:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Verification: PA - LU (should be near zero)
Error: 0.00e+00

Verification: A - P^T @ L @ U (should be near zero)
Error: 0.00e+00


### **3.1 Permutation Matrices**

Permutation matrices are special matrices that represent row or column swaps. A permutation matrix is a square matrix with exactly one 1 in each row and each column, with all other elements being 0. The effect of left-multiplying a matrix by a permutation matrix is to permute (reorder) its rows, while right-multiplication permutes the columns. Permutation matrices appear in LU decomposition because during row reduction, it is sometimes necessary to swap rows to ensure numerical stability or to handle cases where the pivot element is zero. For example, if the diagonal element in the current pivot position is zero or very small, it is standard practice to swap the current row with another row that has a larger pivot element in that column. This row-swapping strategy is called partial pivoting and is crucial for numerical stability. A remarkable property of permutation matrices is that they are orthogonal matrices, meaning P^(-1) = P^T. This follows from the structure of permutation matrices: each column is an orthonormal basis vector with exactly one 1 and zeros elsewhere, so the dot product of any two different columns is 0, and the dot product of a column with itself is 1. Because permutation matrices are orthogonal, they have all the nice numerical properties of orthogonal matrices, including perfect numerical stability. In the context of LU decomposition, the full decomposition is often written as PA = LU or A = P^T LU, where P is the permutation matrix that results from any row swaps performed during the factorization.

In [7]:
P = np.array([[1, 0, 0],
              [0, 0, 1],
              [0, 1, 0]], dtype=float)

# Original matrix
A = np.array([[3, 2, 1],
              [0, 0, 5],
              [0, 7, 2]], dtype=float)

print("Original matrix A:")
print(A)

print("\nPermutation matrix P (swaps rows 1 and 2):")
print(P)

print("\nPA (rows 1 and 2 swapped):")
print(P @ A)

# Verify that P is orthogonal
print("\nVerify P is orthogonal:")
print(f"P^T @ P = I, error: {np.linalg.norm(P.T @ P - np.eye(3)):.2e}")
print(f"P @ P^T = I, error: {np.linalg.norm(P @ P.T - np.eye(3)):.2e}")

# Verify that P^(-1) = P^T
P_inv = np.linalg.inv(P)
print(f"P^(-1) = P^T, error: {np.linalg.norm(P_inv - P.T):.2e}")

Original matrix A:
[[3. 2. 1.]
 [0. 0. 5.]
 [0. 7. 2.]]

Permutation matrix P (swaps rows 1 and 2):
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]

PA (rows 1 and 2 swapped):
[[3. 2. 1.]
 [0. 7. 2.]
 [0. 0. 5.]]

Verify P is orthogonal:
P^T @ P = I, error: 0.00e+00
P @ P^T = I, error: 0.00e+00
P^(-1) = P^T, error: 0.00e+00


## **4. Summary**

This chapter has covered the fundamental techniques of row reduction and LU decomposition, which are cornerstones of numerical linear algebra. Row reduction is a systematic procedure for transforming a matrix into echelon form through elementary row operations, providing the foundation for solving systems of equations. Gaussian elimination combines row reduction with back substitution to solve systems efficiently, and Gauss-Jordan elimination extends this to find the unique reduced row echelon form. LU decomposition factors a matrix A into the product of lower and upper triangular matrices, A = LU (with permutation matrices P when row swaps occur). The LU decomposition is computationally efficient and enables rapid solution of multiple systems with different right-hand sides by reusing the same LU factors. Permutation matrices, which represent row swaps, are orthogonal matrices with the special property that P^(-1) = P^T. LU decomposition has numerous applications including solving systems of equations, computing determinants (as the product of diagonal elements accounting for the permutation), and computing matrix inverses. The key insight is that these factorizations leverage the triangular structure of matrices to enable efficient computation through forward and back substitution, avoiding the numerical instability that can arise from explicitly computing matrix inverses. Understanding LU decomposition is essential for appreciating modern numerical linear algebra and its applications in science and engineering.